# Results for model: meta/llama-3.3-70b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Convert date_id to datetime
df = df.with_column(pl.col('date_id').cast(pl.Int64).alias('date_id'))

# Calculate global TOP_QUANTILE
df = df.with_column(
    pl.col('feature_00').rank(method='dense').alias('feature_00_rank')
)

# Calculate rolling batches of 'date_id'
df = df.with_column(
    pl.col('date_id').alias('date_id_batch')
)

# Calculate TOP_QUANTILE of 'feature_00' relative to 'responder_6' across rolling batches of 'date_id'
top_quantile = df['feature_00'].quantile(0.85)
df = df.with_column(
    (pl.col('feature_00') > top_quantile).alias('top_quantile')
)

# Prepare data for training
X = df[['feature_00', 'top_quantile']]
y = df['responder_6']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Regressor
xgb_model = xgb.XGBRegressor()
xgb_model.fit(X_train, y_train)